# t-SNE para redução de dimensionalidade de expressão gênica

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

## Ler Dados

In [27]:
df_cells = pd.read_csv('./dataset/synthetic_cell_data.csv')
df_cells.head()

,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,gene_10,...,gene_42,gene_43,gene_44,gene_45,gene_46,gene_47,gene_48,gene_49,gene_50,cell_type
0,-2.645739,1.084029,1.033785,-0.492039,2.031826,-1.577436,-3.064917,7.414324,5.415871,-0.189912,...,3.113416,2.623661,-3.358139,4.445828,-1.543928,0.403476,4.233371,2.028571,1.281061,type_4
1,2.501869,-2.607423,0.254079,1.983542,-12.577327,3.643290,0.662070,2.639538,-0.270477,-0.060948,...,-0.713175,-4.108856,12.761196,-0.492549,-0.567835,0.541446,0.951115,5.316995,-2.845845,type_0
2,0.867952,6.024128,-0.025372,-0.639283,-6.976072,-4.469016,-0.079012,-0.494158,-0.422986,-0.397785,...,-0.885718,5.124325,-7.550761,-1.299736,-0.676549,0.895681,2.608146,6.540721,-5.255593,type_1
3,-5.450764,-1.251932,1.479890,-0.006988,7.958121,-6.493759,5.254006,14.731454,1.216928,-0.161186,...,-2.450433,-0.817676,4.400203,3.912871,0.578546,0.101902,-0.678812,1.194707,-0.450927,type_4
4,1.646937,5.899848,1.712686,-0.267201,0.736141,-5.839495,4.971377,20.075666,2.032173,-1.333014,...,0.264370,-14.056835,-0.752921,-2.653600,-1.222040,0.774236,-0.751898,2.663342,11.293108,type_4


In [28]:
df_cells.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 51 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   gene_1     1200 non-null   float64
 1   gene_2     1200 non-null   float64
 2   gene_3     1200 non-null   float64
 3   gene_4     1200 non-null   float64
 4   gene_5     1200 non-null   float64
 5   gene_6     1200 non-null   float64
 6   gene_7     1200 non-null   float64
 7   gene_8     1200 non-null   float64
 8   gene_9     1200 non-null   float64
 9   gene_10    1200 non-null   float64
 10  gene_11    1200 non-null   float64
 11  gene_12    1200 non-null   float64
 12  gene_13    1200 non-null   float64
 13  gene_14    1200 non-null   float64
 14  gene_15    1200 non-null   float64
 15  gene_16    1200 non-null   float64
 16  gene_17    1200 non-null   float64
 17  gene_18    1200 non-null   float64
 18  gene_19    1200 non-null   float64
 19  gene_20    1200 non-null   float64
 20  gene_21    1200 non

In [29]:
df_cells.describe()

,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,gene_10,...,gene_41,gene_42,gene_43,gene_44,gene_45,gene_46,gene_47,gene_48,gene_49,gene_50
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,...,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,0.269324,2.026219,0.015786,-0.028116,-2.122801,-0.490857,1.284298,5.828234,0.362583,0.011576,...,0.365303,0.228197,-4.467380,0.869411,0.351452,0.024154,-0.021449,-0.298270,1.336862,0.386345
std,3.775422,3.047931,1.019325,1.010042,10.841627,3.609355,3.468614,10.981872,3.652475,0.973866,...,3.694792,3.585006,12.248655,9.529351,3.733268,1.019383,1.010276,3.681659,3.565798,3.550924
min,-10.141897,-9.228910,-3.482768,-3.242987,-41.441590,-10.793123,-8.996350,-27.318668,-11.678312,-3.003106,...,-11.960700,-10.366488,-38.545817,-36.738775,-12.783092,-3.246233,-2.999714,-13.125571,-10.672461,-10.133236
25%,-2.421130,-0.046719,-0.649693,-0.686873,-9.392613,-2.998869,-1.287978,-1.725341,-2.108999,-0.648617,...,-2.127106,-2.097078,-12.919963,-5.672383,-2.275796,-0.671541,-0.702535,-2.838792,-1.001999,-2.067543
50%,0.184105,2.071347,0.015290,-0.046073,-2.297948,-0.570461,1.404739,5.651884,0.191376,0.007060,...,0.385808,0.083860,-4.561076,0.866118,0.198840,0.015510,-0.022558,-0.379001,1.540704,0.241216
75%,2.889395,4.047296,0.744118,0.629383,4.491653,1.843312,3.737431,13.200223,2.845956,0.700536,...,2.876378,2.840972,3.336691,7.170686,3.000446,0.720197,0.688822,2.221177,3.875756,2.864945
max,12.232814,13.086474,3.475239,3.175204,41.637061,10.616334,11.972265,46.421128,11.404758,3.076787,...,12.015285,10.727785,34.017735,39.138959,11.778587,3.313657,3.374655,11.040306,11.612362,12.975577


## Tratamento dos dados

In [30]:
colunas_numericas = df_cells.select_dtypes('number').columns.to_list()
trasnformer_numerico = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
  ('num', trasnformer_numerico, colunas_numericas)
])

In [31]:
X = df_cells.drop(columns=['cell_type'])
y = df_cells['cell_type']

X_transformed = preprocessor.fit_transform(X=X)
print(f"Shape: {X_transformed.shape}")
print(f"X_transformed:\n{X_transformed}")

Shape: (1200, 50)
X_transformed:
[[-0.77243764 -0.30925347  0.99911555 ...  1.23138234  0.19406509
   0.25207215]
 [ 0.59158306 -1.52089217  0.23387275 ...  0.33949527  1.11666232
  -0.91061889]
 [ 0.15862546  1.31222629 -0.04039477 ...  0.78976021  1.45998971
  -1.58952741]
 ...
 [-1.1511887   0.29350813 -0.53687154 ... -0.71829898  0.49844215
   0.31968906]
 [ 0.58742701 -0.31050335 -1.6682397  ... -0.47976208  0.4266841
  -0.01311075]
 [ 0.47010141 -1.13816167 -0.33354413 ...  1.60336574  0.14627282
  -0.31349598]]


## Modelo 2D

In [38]:
perplexities = list(range(5, 20+1))
df_results = pd.DataFrame()

for perplexity in perplexities:
    print(perplexity)
    tsne = TSNE(
        init='random',
        max_iter=250,
        perplexity=perplexity,
        random_state=51,
        n_components=2,
    )

    tsne_result = tsne.fit_transform(X=X_transformed)

    temp_df = pd.DataFrame(tsne_result, columns=['Component1', 'Component2'])
    temp_df['Perplexity'] = perplexity
    temp_df = pd.concat([temp_df, y], axis=1)
    df_results = pd.concat([df_results, temp_df], ignore_index=True)


5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20


## Modelo 3D

In [ ]:
perplexities = list(range(5, 20+1))
df_results3 = pd.DataFrame()

for perplexity in perplexities:
    print(perplexity)
    tsne = TSNE(
        init='random',
        max_iter=250,
        perplexity=perplexity,
        random_state=51,
        n_components=3,
    )

    tsne_result = tsne.fit_transform(X=X_transformed)

    temp_df3 = pd.DataFrame(tsne_result, columns=['Component1', 'Component2', 'Component3'])
    temp_df3['Perplexity'] = perplexity
    temp_df3 = pd.concat([temp_df3, y], axis=1)
    df_results3 = pd.concat([df_results3, temp_df3], ignore_index=True)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20


## Visualizar Resultado 2D

In [39]:
import plotly.io as pio
import plotly.express as px

pio.renderers.default = 'browser'

fig = px.scatter(
  df_results,
  x='Component1',
  y='Component2',
  animation_frame='Perplexity',
  color='cell_type',
  title="Visualização do t-SNE com variação do Perplexity - 2D"
)
fig.show()

## Visualizar Resultado 3D

In [37]:
fig = px.scatter_3d(
  df_results3,
  x='Component1',
  y='Component2',
  z='Component3',
  animation_frame='Perplexity',
  color='cell_type',
  title="Visualização do t-SNE com variação do Perplexity - 3D"
)
fig.show()